# CUTLASS 教程：导读

> 本教程由微信公众号文章《CUTLASS 笔记上半部分（转载）》整理而成，原文转载自知乎专栏 [CUTLASS 笔记](https://www.zhihu.com/column/c_1938664963049763058)，已获作者授权。
> 原文链接：https://mp.weixin.qq.com/s/aXOCvfPGW_XoX03Q9oakmw

## 这个系列讲什么？

从一个最小的 Minimal GEMM 开始，逐步扩展 CuTe、CUTLASS 各类组件和 Hopper、Blackwell 等新架构的特性，最终实现一个高性能的 GEMM 融合算子。本目录只整理了"上半部分"：SM80 及之前架构的 CuTe 基础 + 三级 Tiling。

## 1. 为什么学 CUTLASS？

CUTLASS 广泛应用于 Pytorch、vLLM、FA2、FA3 等常用框架和算子库。它的学习曲线陡峭，但**可控且灵活**：

- CUTLASS 是**完全白盒**的：本质是对 PTX 指令的封装，不需要猜测 CUDA Runtime API 背后做了什么；
- 写代码时就能感知硬件会如何执行，新指令集出来后可以自己扩展；
- 学会 CUTLASS 本质上就是在学 CUDA / PTX，即使以后写 triton，有了这套知识储备也能得心应手。

### 2025 年写高性能算子的三条路径

| 路径 | 代表 | 特点 |
|---|---|---|
| 自动编译 | torch.compile | 开发快，控制力弱 |
| Python DSL + PTX 编译器 | triton、TileLang、Mojo | 效率与灵活性折中 |
| C++ 模板封装 PTX | **CUTLASS**、Thunder Kittens | 可控、灵活、贴近硬件 |

![CUTLASS GEMM Hierarchy](assets/figs/fig_01.png)

## 2. 前置知识清单

- NV GPU 编程模型：线程层级（grid / block / warp / thread）、内存层级（GMEM / SMEM / Register）；
- Python 与 C++ 基础语法，最好能读懂 C++17 模板语法；
- CUDA C++ 扩展语法：`__device__` / `__global__` 区别、`threadIdx` / `blockIdx` 用法；
- 会写一个简单 CUDA kernel（例如 element-wise 向量加）。

## 3. 本目录的组织方式（对照原文笔记）

| 本目录文件 | 对应原文 | 主题 |
|---|---|---|
| `01_SM架构与Tensor_Core.md` | Extra | SM 架构、CUDA Core 与 Tensor Core 前置知识 |
| `02_CuTe基础与Minimal_GEMM.md` | 笔记 (1) | CuTe 组件 + 单 MMA 指令的 Minimal GEMM + 验证/NCU/PTX |
| `03_混合精度GEMM与FP8.md` | 笔记 (2) | 混合精度、精度转换、自定义 FP8 MMA op、TV/MN Layout |
| `04_Tiled_MMA与三级Tiling.md` | 笔记 (3) | 优化方法论、三级 Tiling、make_tiled_mma |
| `05_Tiled_Copy与全局内存访存.md` | 笔记 (4) | 向量化/合并访存、TiledCopy 原理 |
| `06_Block_MMA.md` | 笔记 (5) | Tile 扩展到 Block、循环 Copy/MMA |
| `07_Block_Copy与SMEM.md` | 笔记 (6) | SMEM 特性、二级拷贝、Bank Conflict、NCU 分析 |

每课末尾有思考题，学完把答案发给我，我继续审查——和 Triton 课程一个节奏。

## 4. 原文推荐的教程

- **@reed**：中文社区最好的 CUTLASS 教程；
- **Colfax Research**（https://research.colfax-intl.com/blog/）：FA3 主要开发团队，最好的外网教程；
- **@进击的Killua**：代码示例和图片讲解丰富，适合新手；
- **@Anonymous**：对 API 和 PTX 指令细节研究深入；
- **CUTLASS Discussions**（https://github.com/NVIDIA/cutlass/discussions）：官方开发者积极回复。

## 5. 环境与版本

- 原文使用 CUTLASS **4.1.0**（笔记 1–5）、**4.3.4**（笔记 6），硬件架构 **SM90**（Hopper）；
- 系列代码已开源：**cutlass-notes**（GitHub 仓库，原文中有链接）；
- 本目录配图存放在 `assets/figs/`，与原文图号对应。